# Лабораторная работа: CIFAR-10 ResNet

Запустите ячейки сверху вниз. Полученные метрики внесите в вывод.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
keras.utils.set_random_seed(42)
print('GPU:',tf.config.list_physical_devices('GPU'))

In [ ]:
(a,c),(b,d)=keras.datasets.cifar10.load_data()
a=a.astype('float32')/255;b=b.astype('float32')/255
def block(x,n,stride=1):
    skip=x
    x=layers.Conv2D(n,3,strides=stride,padding='same',use_bias=False)(x)
    x=layers.BatchNormalization()(x);x=layers.Activation('relu')(x)
    x=layers.Conv2D(n,3,padding='same',use_bias=False)(x);x=layers.BatchNormalization()(x)
    if stride!=1 or skip.shape[-1]!=n:skip=layers.Conv2D(n,1,strides=stride,padding='same')(skip)
    return layers.Activation('relu')(layers.Add()([x,skip]))
inp=keras.Input((32,32,3));x=layers.Conv2D(32,3,padding='same')(inp)
for n,s in [(32,1),(32,1),(64,2),(64,1),(128,2),(128,1)]:x=block(x,n,s)
out=layers.Dense(10,activation='softmax')(layers.GlobalAveragePooling2D()(x))
model=keras.Model(inp,out);model.compile('adam','sparse_categorical_crossentropy',metrics=['accuracy'])
history=model.fit(a,c,validation_split=.1,epochs=15,batch_size=128)
pd.DataFrame(history.history).plot(grid=True);plt.show()
print('Test:',model.evaluate(b,d))